#### Autora: Giulia Katherine Romas Tomazeli

# Bibliotecas

In [1]:
from pathlib import Path
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import sys
import pandas as pd
import time

In [2]:
PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

In [3]:
from configs.config import (
    DATA_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,
    IMAGE_SIZE,
    BATCH_SIZE,
    NUM_WORKERS,
    NUM_CLASSES,
    CLASS_NAMES,
    MEAN,
    STD,
    NUM_EPOCHS,
    EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_MIN_DELTA,
    PRETRAINED,
    FREEZE_BACKBONE,
    DEVICE,
    SEED,
    TRAIN_DIR
)

Device: cuda


In [ ]:
from utils.dataset import NascentesDataset, samples_to_dataframe
from utils.spatial_kfold import (load_spatial_groups, create_spatial_folds)
from utils.train import train_model, train_final_model, calculate_class_weights
from utils.early_stopping import EarlyStopping
from utils.evaluate import evaluate_model
from utils.plots import (plot_training_history, plot_f1_history, plot_confusion_matrix, save_misclassified_images)
from utils.logger import get_logger
from utils.metrics import print_metrics, save_metrics

from models.convnext_swin import create_model

c:\Users\giuli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.2)
  from scipy.sparse import csr_matrix, issparse


# Configurações

In [5]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
MODEL_NAME = "final_model"
N_SPLITS = 5

print(f"Modelo: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Folds: {N_SPLITS}")

Modelo: final_model
Device: cuda
Folds: 5


In [7]:
SHAPEFILE_PATH = PROJECT_DIR / "data" / "amostras" / "amostra_final.shp"

In [8]:
CHECKPOINT_FINAL = PROJECT_DIR / "checkpoints" / MODEL_NAME / "final_model.pth"

In [9]:
result_dir_test = RESULTS_DIR / MODEL_NAME / "test"
result_dir_test.mkdir(parents=True, exist_ok=True)

# Dataset

In [30]:
train_dataset = NascentesDataset(root_dir=TRAIN_DIR, class_to_idx={"Preservada": 0, "Moderada": 1, "Baixa": 2, "Não Preservada": 3}, image_size=IMAGE_SIZE, mean=MEAN, std=STD, augmentation=True)
val_dataset = NascentesDataset(root_dir=TRAIN_DIR, class_to_idx={"Preservada": 0, "Moderada": 1, "Baixa": 2, "Não Preservada": 3}, image_size=IMAGE_SIZE, mean=MEAN, std=STD, augmentation=False)

In [31]:
samples_df = samples_to_dataframe(train_dataset)

print(f"Total de imagens: {len(samples_df)}")
samples_df.head()

Total de imagens: 1538


,path,label,fid
0,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1004354
1,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1004578
2,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1016991
3,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1019727
4,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_103701


In [32]:
dataset_fids = samples_df["fid"].tolist()

# Spatial K-Fold

In [33]:
spatial_groups = load_spatial_groups(shapefile_path=SHAPEFILE_PATH, dataset_fids=dataset_fids)

spatial_groups.head()

,fid,ugrhi_id,ugrhi_nome
0,1365920,20,Aguapeí
1,764554,20,Aguapeí
2,639552,20,Aguapeí
4,1894144,20,Aguapeí
5,1714024,20,Aguapeí


In [34]:
samples_with_groups, folds = create_spatial_folds(samples=samples_df, spatial_groups=spatial_groups, n_splits=N_SPLITS, random_state=SEED)

In [35]:
for fold in folds:
    print(f"Fold {fold['fold']}: treino={len(fold['train_idx'])}, validação={len(fold['val_idx'])}")

Fold 1: treino=1192, validação=346
Fold 2: treino=1265, validação=273
Fold 3: treino=1181, validação=357
Fold 4: treino=1258, validação=280
Fold 5: treino=1256, validação=282


## Verificação dos folds

In [36]:
for fold in folds:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]

    print(f"\n=-=-= Fold {fold['fold']} =-=-=")

    print("Treino:")
    print(samples_with_groups.iloc[train_idx]["label"].value_counts(normalize=True).sort_index())

    print("\nValidação:")
    print(samples_with_groups.iloc[val_idx]["label"].value_counts(normalize=True).sort_index())


=-=-= Fold 1 =-=-=
Treino:
label
0    0.387584
1    0.216443
2    0.248322
3    0.147651
Name: proportion, dtype: float64

Validação:
label
0    0.320809
1    0.190751
2    0.268786
3    0.219653
Name: proportion, dtype: float64

=-=-= Fold 2 =-=-=
Treino:
label
0    0.369960
1    0.207905
2    0.246640
3    0.175494
Name: proportion, dtype: float64

Validação:
label
0    0.384615
1    0.223443
2    0.282051
3    0.109890
Name: proportion, dtype: float64

=-=-= Fold 3 =-=-=
Treino:
label
0    0.351397
1    0.209145
2    0.266723
3    0.172735
Name: proportion, dtype: float64

Validação:
label
0    0.442577
1    0.215686
2    0.207283
3    0.134454
Name: proportion, dtype: float64

=-=-= Fold 4 =-=-=
Treino:
label
0    0.373609
1    0.206677
2    0.255962
3    0.163752
Name: proportion, dtype: float64

Validação:
label
0    0.367857
1    0.228571
2    0.239286
3    0.164286
Name: proportion, dtype: float64

=-=-= Fold 5 =-=-=
Treino:
label
0    0.379777
1    0.213376
2    0.247611
3   

In [37]:
for fold in folds:
    train_ugrhi = set(samples_with_groups.iloc[fold["train_idx"]]["ugrhi_id"])
    val_ugrhi = set(samples_with_groups.iloc[fold["val_idx"]]["ugrhi_id"])

    overlap = train_ugrhi.intersection(val_ugrhi)

    print(f"Fold {fold['fold']} - UGRHIs compartilhadas: {len(overlap)}")

Fold 1 - UGRHIs compartilhadas: 0
Fold 2 - UGRHIs compartilhadas: 0
Fold 3 - UGRHIs compartilhadas: 0
Fold 4 - UGRHIs compartilhadas: 0
Fold 5 - UGRHIs compartilhadas: 0


# Treinamento e validação

In [38]:
lr = 1e-4
weight_decay = 5e-2
dropout = 0.4
hidden_dim = 512

In [39]:
fold_results = []

In [40]:
for fold_info in folds:
    fold = fold_info["fold"]

    logger = get_logger(MODEL_NAME, f"fold_{fold}")

    logger.info("=-" * 30)
    logger.info(f"FOLD {fold}/{N_SPLITS}")
    logger.info("=-" * 30)

    train_idx = fold_info["train_idx"]
    val_idx = fold_info["val_idx"]

    # -=-=-=-=- subsets -=-=-=-=- #
    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(val_dataset, val_idx)

    # -=-=-=-=- dataloaders -=-=-=-=- #
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    
    model = create_model(num_classes=NUM_CLASSES, pretrained=PRETRAINED, freeze_backbone=FREEZE_BACKBONE, hidden_dim=hidden_dim, dropout=dropout)
    model = model.to(DEVICE)

    # -=-=-=-=- pesos das classes -=-=-=-=- #
    class_weights, class_counts = calculate_class_weights(train_subset, NUM_CLASSES)

    class_weights = class_weights.to(DEVICE)

    print(f"\nContagem das classes: {class_counts}")
    print(f"Pesos das classes: {class_weights.cpu().numpy()}\n")

    # -=-=-=-=- funcao de perda e otimizador -=-=-=-=- #
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    
    checkpoint_dir = (CHECKPOINT_DIR / MODEL_NAME / f"fold_{fold}")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = (checkpoint_dir / "best_model.pth")

    early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, min_delta=EARLY_STOPPING_MIN_DELTA, mode="min", save_path=checkpoint_path)
    
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        early_stopping=early_stopping,
        device=DEVICE,
        num_epochs=NUM_EPOCHS,
        class_names=CLASS_NAMES,
        logger=logger,
        freeze_backbone=FREEZE_BACKBONE
    )

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))

    result_dir = RESULTS_DIR / MODEL_NAME / f"fold_{fold}"
    result_dir.mkdir(parents=True, exist_ok=True)

    errors_df = save_misclassified_images(
        model=model,
        dataloader=val_loader,
        device=DEVICE,
        class_names=CLASS_NAMES,
        output_dir=result_dir / "errors_val"
    )

    errors_df.to_csv(result_dir / "misclassified.csv", index=False)

    best_epoch = np.argmin(history["val_loss"])

    best_val_loss = history["val_loss"][best_epoch]
    best_val_f1 = history["val_f1"][best_epoch]
    best_val_accuracy = history["val_accuracy"][best_epoch]
    best_val_precision = history["val_precision_macro"][best_epoch]
    best_val_recall = history["val_recall_macro"][best_epoch]

    print(f"-=-=-=-=-=-=- Fold {fold} -=-=-=-=-=-=-")
    print(f"Melhor época: {best_epoch + 1}\n")
    print(f"Melhor F1-macro: {best_val_f1:.4f}")
    print(f"Melhor loss: {best_val_loss:.4f}")
    print(f"Melhor accuracy: {best_val_accuracy:.4f}")
    print(f"Melhor precision: {best_val_precision:.4f}")
    print(f"Melhor recall: {best_val_recall:.4f}")

    plot_training_history(history, result_dir / "loss.png")
    plot_f1_history(history, result_dir / "f1.png")

    fold_results.append({"fold": fold, "best_epoch": best_epoch + 1, "best_val_loss": best_val_loss, "val_accuracy": best_val_accuracy, "val_precision_macro": best_val_precision, "val_recall_macro": best_val_recall, "val_f1_macro": best_val_f1})


Contagem das classes: [462 258 296 176]
Pesos das classes: [0.6450216 1.1550387 1.0067568 1.6931819]



-=-=-=-=-=-=- Fold 1 -=-=-=-=-=-=-
Melhor época: 11

Melhor F1-macro: 0.7075
Melhor loss: 0.5864
Melhor accuracy: 0.7110
Melhor precision: 0.7228
Melhor recall: 0.7133

Contagem das classes: [468 263 312 222]
Pesos das classes: [0.6757479 1.2024715 1.0136218 1.4245496]



-=-=-=-=-=-=- Fold 2 -=-=-=-=-=-=-
Melhor época: 23

Melhor F1-macro: 0.6372
Melhor loss: 0.7577
Melhor accuracy: 0.6923
Melhor precision: 0.6458
Melhor recall: 0.6324

Contagem das classes: [415 247 315 204]
Pesos das classes: [0.7114458 1.1953441 0.9373016 1.4473039]



-=-=-=-=-=-=- Fold 3 -=-=-=-=-=-=-
Melhor época: 8

Melhor F1-macro: 0.7250
Melhor loss: 0.6139
Melhor accuracy: 0.7619
Melhor precision: 0.7321
Melhor recall: 0.7208

Contagem das classes: [470 260 322 206]
Pesos das classes: [0.6691489  1.2096153  0.97670805 1.5266991 ]



-=-=-=-=-=-=- Fold 4 -=-=-=-=-=-=-
Melhor época: 9

Melhor F1-macro: 0.6826
Melhor loss: 0.6791
Melhor accuracy: 0.7071
Melhor precision: 0.6804
Melhor recall: 0.6949

Contagem das classes: [477 268 311 200]
Pesos das classes: [0.6582809 1.1716418 1.0096463 1.57     ]



-=-=-=-=-=-=- Fold 5 -=-=-=-=-=-=-
Melhor época: 10

Melhor F1-macro: 0.6806
Melhor loss: 0.6052
Melhor accuracy: 0.6915
Melhor precision: 0.6791
Melhor recall: 0.6902


## Resultados dos folds

In [41]:
fold_results_df = pd.DataFrame(fold_results)
fold_results_df

,fold,best_epoch,best_val_loss,val_accuracy,val_precision_macro,val_recall_macro,val_f1_macro
0,1,11,0.586361,0.710983,0.722777,0.713329,0.707453
1,2,23,0.757700,0.692308,0.645766,0.632404,0.637229
2,3,8,0.613879,0.761905,0.732059,0.720830,0.725012
3,4,9,0.679127,0.707143,0.680359,0.694942,0.682637
4,5,10,0.605154,0.691489,0.679107,0.690219,0.680600


In [42]:
metrics = ["best_val_loss", "val_accuracy", "val_precision_macro", "val_recall_macro", "val_f1_macro"]

In [43]:
summary = pd.DataFrame({"Média": fold_results_df[metrics].mean(), "Desvio padrão": fold_results_df[metrics].std()})
print(summary)

                        Média  Desvio padrão
best_val_loss        0.648444       0.070359
val_accuracy         0.712765       0.028813
val_precision_macro  0.692014       0.035324
val_recall_macro     0.690345       0.034770
val_f1_macro         0.686586       0.033164


# Treinamento final

## Dataset final

In [52]:
final_train_dataset = NascentesDataset(root_dir=DATA_DIR / "treino", class_to_idx={"Preservada": 0, "Moderada": 1, "Baixa": 2, "Não Preservada": 3}, image_size=IMAGE_SIZE, mean=MEAN, std=STD, augmentation=True)

In [53]:
final_train_loader = DataLoader(final_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

## Modelo

In [54]:
model = create_model(num_classes=NUM_CLASSES, pretrained=PRETRAINED, freeze_backbone=FREEZE_BACKBONE)
model = model.to(DEVICE)

## Configurações finais

In [55]:
final_epochs = int(fold_results_df["best_epoch"].max())

In [56]:
final_epochs

23

In [57]:
class_weights, class_counts = calculate_class_weights(final_train_dataset, NUM_CLASSES)
class_weights = class_weights.to(DEVICE)

In [58]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)

## Treino

In [59]:
logger = get_logger(MODEL_NAME,"final")

final_history, training_time = train_final_model(
    model=model,
    train_loader=final_train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    num_epochs=final_epochs,
    class_names=CLASS_NAMES,
    model_name=MODEL_NAME,
    checkpoint_dir=CHECKPOINT_DIR,
    results_dir=RESULTS_DIR,
    logger=logger
)

# Teste

In [10]:
model = create_model(num_classes=NUM_CLASSES, pretrained=PRETRAINED, freeze_backbone=FREEZE_BACKBONE)
model.load_state_dict(torch.load(CHECKPOINT_FINAL, map_location=DEVICE))

model = model.to(DEVICE)

In [11]:
test_dataset = NascentesDataset(root_dir=DATA_DIR / "teste", class_to_idx={"Preservada": 0, "Moderada": 1, "Baixa": 2, "Não Preservada": 3}, image_size=IMAGE_SIZE, mean=MEAN, std=STD)

In [12]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## Resultados dos testes

In [13]:
metrics, y_true, y_pred, fids = evaluate_model(model=model, loader=test_loader, device=DEVICE, class_names=CLASS_NAMES)

In [78]:
df_errors = save_misclassified_images(
    model=model,
    dataloader=test_loader,
    device=DEVICE,
    class_names=CLASS_NAMES,
    output_dir=RESULTS_DIR / MODEL_NAME / "errors_test"
)

In [ ]:
plot_confusion_matrix(y_true, y_pred, ["Conservada", "Moderada", "Baixa", "Não Conservada"], RESULTS_DIR / MODEL_NAME / "test/test_confusion_matrix.png")

In [80]:
save_metrics(metrics=metrics, class_names=CLASS_NAMES, save_path=result_dir_test / "metrics.csv")

In [81]:
print_metrics(metrics, CLASS_NAMES)


=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
RESULTADOS
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Acurácia:        0.7500
Precision macro: 0.7368
Recall macro:    0.7403
F1-macro:        0.7380

Métricas por classe:
Classe         Precision      Recall          F1     Support
------------------------------------------------------------
Preservada        0.8601      0.8913      0.8754         138
Moderada          0.6500      0.6566      0.6533          99
Baixa             0.6706      0.6064      0.6369          94
Não Preservada      0.7667      0.8070      0.7863          57
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-


In [82]:
real_counts = pd.Series(y_true).value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0)
pred_counts = pd.Series(y_pred).value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0)

# Porcentagens
real_pct = (real_counts / len(y_true)) * 100
pred_pct = (pred_counts / len(y_pred)) * 100

# Tabela
df_distribuicao = pd.DataFrame({
    "Classe": CLASS_NAMES,
    "Real_n": real_counts.values,
    "Real_%": real_pct.values,
    "Predito_n": pred_counts.values,
    "Predito_%": pred_pct.values
})

print(df_distribuicao.to_string(index=False))

        Classe  Real_n    Real_%  Predito_n  Predito_%
    Preservada     138 35.567010        143  36.855670
      Moderada      99 25.515464        100  25.773196
         Baixa      94 24.226804         85  21.907216
Não Preservada      57 14.690722         60  15.463918


# Contagem dos parâmetros

In [83]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parâmetros totais: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")
print(f"Parâmetros totais (milhões): {total_params / 1e6:.2f} M")
print(f"Parâmetros treináveis (milhões): {trainable_params / 1e6:.2f} M")

Parâmetros totais: 56,329,696
Parâmetros treináveis: 990,214
Parâmetros totais (milhões): 56.33 M
Parâmetros treináveis (milhões): 0.99 M
